In [ ]:
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18


ROOT = Path(".")
TRAIN_CSV = ROOT / "dataset" / "train_data" / "train_data.csv"
TEST_CSV = ROOT / "dataset" / "test_data" / "test_data.csv"
TRAIN_IMAGE_DIR = ROOT / "dataset" / "train_data" / "images"
TEST_IMAGE_DIR = ROOT / "dataset" / "test_data" / "images"
OUTPUT_CSV = ROOT / "submission.csv"
BATCH_SIZE = 64

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
image_columns = ["img1", "img2", "img3"]

test_image_names = pd.unique(test_df[image_columns].stack().to_numpy())
train_image_names = train_df["image"].to_numpy()

test_df.groupby("subtaskID").size(), len(train_image_names), len(test_image_names)


In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

model = resnet18(weights=weights)
model.eval().to(device)


class ImageNameDataset(Dataset):
    def __init__(self, image_names, image_dir, transform):
        self.image_names = list(image_names)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image = Image.open(self.image_dir / image_name).convert("RGB")
        return image_name, self.transform(image)


def compute_logits(image_names, image_dir, label):
    dataset = ImageNameDataset(image_names, image_dir, preprocess)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    logits_by_image = {}

    with torch.inference_mode():
        for batch_idx, (names, images) in enumerate(loader, start=1):
            logits = model(images.to(device)).cpu()
            logits_by_image.update(zip(names, logits))
            if batch_idx == 1 or batch_idx == len(loader) or batch_idx % 10 == 0:
                print(f"{label}: {batch_idx}/{len(loader)} batches", flush=True)

    return logits_by_image


In [ ]:
train_logits_by_image = compute_logits(train_image_names, TRAIN_IMAGE_DIR, "train")
test_logits_by_image = compute_logits(test_image_names, TEST_IMAGE_DIR, "test")

len(train_logits_by_image), len(test_logits_by_image)


In [ ]:
train_logit_matrix = torch.stack([train_logits_by_image[name] for name in train_df["image"]])
train_angles = torch.tensor(train_df["angle"].to_numpy(), dtype=torch.long)


def find_intruder(image_names):
    vectors = torch.stack([test_logits_by_image[name] for name in image_names])
    distances = torch.cdist(vectors, vectors, p=2)
    return int(torch.argmax(distances.sum(dim=1)).item()) + 1


def predict_angle(image_name):
    vector = test_logits_by_image[image_name].unsqueeze(0)
    nearest_idx = torch.cdist(vector, train_logit_matrix, p=2).argmin().item()
    return int(train_angles[nearest_idx].item())


In [ ]:
answers = []

for row in test_df.itertuples(index=False):
    if row.subtaskID == 1:
        answers.append(find_intruder((row.img1, row.img2, row.img3)))
    else:
        answers.append(predict_angle(row.img1))

submission = test_df[["subtaskID", "datapointID"]].copy()
submission["answer"] = answers
submission.to_csv(OUTPUT_CSV, index=False)

submission.head(), submission.tail(), OUTPUT_CSV
